# Lab 4C: Track experiments with MLflow (10 minutes)

**Presenter: Nagarjuna Naidu** | Day 4 | about 10 minutes

- Log params, metrics and models for three runs, then compare them
- Register the best model and promote it with an alias
- Check drift with PSI and decide a model swap with an A/B test

## 1. Setup and data: synthetic loan defaults (about 12% positive)

In [ ]:
import os
for k in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[k] = "1"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
import shutil, time, logging, warnings
import numpy as np, pandas as pd
import mlflow, mlflow.sklearn
from mlflow import MlflowClient
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
warnings.filterwarnings("ignore")
for n in ("mlflow", "alembic", "urllib3"):
    logging.getLogger(n).setLevel(logging.ERROR)
INK, TEAL, LIME, CORAL, GOLD, PURPLE = "#0B1F2A", "#0E7C86", "#9BD13B", "#E4572E", "#F2B134", "#6C4AB6"

rng = np.random.default_rng(42)
n = 3000
X = pd.DataFrame({
    "income_k": rng.normal(60, 15, n),
    "debt_ratio": rng.beta(2, 5, n),
    "late_payments": rng.poisson(1.0, n),
    "age": rng.normal(40, 11, n),
})
z = (-0.05 * (X.income_k - 60) + 4.0 * (X.debt_ratio - 0.29) + 0.6 * (X.late_payments - 1)
     - 0.02 * (X.age - 40) + rng.normal(0, 1, n))
y = (z > np.quantile(z, 0.88)).astype(int)          # top 12% become defaults
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
print(X.shape, f"positive rate {y.mean():.1%}", "| train", len(X_tr), "test", len(X_te))

# Local store only: sqlite database + artifact folder under artifacts/ (reset on every run)
shutil.rmtree("artifacts/mlruns", ignore_errors=True)
for f in ("artifacts/mlflow.db",):
    if os.path.exists(f): os.remove(f)
os.makedirs("artifacts/mlruns", exist_ok=True)
mlflow.set_tracking_uri("sqlite:///artifacts/mlflow.db")
EXP = f"loan_default_{int(time.time())}"
exp_id = mlflow.create_experiment(EXP, artifact_location="./artifacts/mlruns/" + EXP)
mlflow.set_experiment(experiment_id=exp_id)
print("MLflow", mlflow.__version__, "| experiment", EXP)

## 2. Log three runs, compare, pick the best

In [ ]:
MODELS = {
    "logreg": (make_pipeline(StandardScaler(), LogisticRegression(max_iter=500)), {"C": 1.0}),
    "random_forest": (RandomForestClassifier(n_estimators=100, max_depth=6, n_jobs=1, random_state=42), {"n_estimators": 100, "max_depth": 6}),
    "hist_gb": (HistGradientBoostingClassifier(max_iter=100, learning_rate=0.1, max_depth=3, random_state=42), {"max_iter": 100, "learning_rate": 0.1}),
}

def log_model_compat(model, X_example):
    """Handles MLflow API differences: name= (3.x) vs artifact_path= (2.x); skops default (3.1x) vs pickle."""
    kw = dict(input_example=X_example, serialization_format="cloudpickle")  # our own models: trusted
    try:
        return mlflow.sklearn.log_model(model, name="model", **kw)
    except TypeError:
        return mlflow.sklearn.log_model(model, artifact_path="model", **kw)

RUNS = {}
for name, (model, params) in MODELS.items():
    with mlflow.start_run(run_name=name) as run:
        model.fit(X_tr, y_tr)
        p = model.predict_proba(X_te)[:, 1]
        mlflow.log_param("model", name)
        mlflow.log_params(params)
        mlflow.log_metric("roc_auc", roc_auc_score(y_te, p))
        mlflow.log_metric("pr_auc", average_precision_score(y_te, p))
        log_model_compat(model, X_tr.head(3))
        RUNS[name] = run.info.run_id

In [ ]:
runs = mlflow.search_runs(experiment_ids=[exp_id], order_by=["metrics.roc_auc DESC"])
print(runs[["tags.mlflow.runName", "metrics.roc_auc", "metrics.pr_auc"]].round(3)
      .rename(columns=lambda c: c.split(".")[-1]).to_string(index=False))
best = runs.iloc[0]
BEST_RUN, BEST_NAME = best["run_id"], best["tags.mlflow.runName"]
print("best:", BEST_NAME, "| run", BEST_RUN[:8])

## 3. Registry: register, alias "production", load back

In [ ]:
REG = "loan_default_model"
client = MlflowClient()
try:
    mv = mlflow.register_model(f"runs:/{BEST_RUN}/model", REG)
except Exception:  # MLflow 3 can store logged models separately: use the model id
    lm = mlflow.search_logged_models(experiment_ids=[exp_id], output_format="list")
    lm = [m for m in lm if m.source_run_id == BEST_RUN][0]
    mv = mlflow.register_model(f"models:/{lm.model_id}", REG)
client.set_registered_model_alias(REG, "production", mv.version)   # aliases replace stages
print("registered", REG, "version", mv.version, "| alias production ->",
      client.get_model_version_by_alias(REG, "production").version)

loaded = mlflow.sklearn.load_model(f"models:/{REG}@production")
print(pd.DataFrame({"default_prob": loaded.predict_proba(X_te.head(5))[:, 1].round(3),
                    "actual": y_te[:5] if isinstance(y_te, np.ndarray) else y_te.values[:5]}).to_string(index=False))

## 4. Monitor and decide

In [ ]:
def psi(ref, cur, bins=10):
    """Population Stability Index with decile bins taken from the reference sample."""
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    r = np.histogram(ref, edges)[0] / len(ref)
    c = np.histogram(cur, edges)[0] / len(cur)
    r, c = np.clip(r, 1e-4, None), np.clip(c, 1e-4, None)
    return float(np.sum((c - r) * np.log(c / r)))

prod = X_te.copy()
prod["income_k"] = prod["income_k"] - 12                    # simulated shift: lower incomes in production
prod["debt_ratio"] = prod["debt_ratio"] * 1.15
rows = []
for col in X_tr.columns:
    v = psi(X_tr[col].values, prod[col].values)
    rows.append((col, round(v, 3), "stable" if v < 0.1 else "watch" if v < 0.25 else "RETRAIN"))
print(pd.DataFrame(rows, columns=["feature", "psi", "verdict"]).to_string(index=False))
print("rule: PSI < 0.1 stable | 0.1-0.25 watch | > 0.25 significant drift, investigate and retrain")

Offline metrics suggest; online tests decide.

In [ ]:
from scipy.stats import norm
a_conv, a_n, b_conv, b_n = 520, 5000, 570, 5000             # simulated conversions: current vs candidate
pa, pb = a_conv / a_n, b_conv / b_n
pool = (a_conv + b_conv) / (a_n + b_n)
se = np.sqrt(pool * (1 - pool) * (1 / a_n + 1 / b_n))
zstat = (pb - pa) / se
pval = 2 * (1 - norm.cdf(abs(zstat)))
print(f"A {pa:.2%} | B {pb:.2%} | lift {pb - pa:+.2%} ({(pb / pa - 1):+.1%} relative) | z {zstat:.2f} | p {pval:.3f}")
print("rule: ship B only if p < 0.05 and lift > 0 ->",
      "SHIP B" if pval < 0.05 and pb > pa else "KEEP A / keep testing")